In [0]:
from pyspark.sql.functions import count

data = [
    (1, "A"),
    (2, "B"),
    (1, "A"),
    (3, "C"),
    (2, "B"),
    (4, "D")
]

df = spark.createDataFrame(data, ["id", "name"])
df.show()

+---+----+
| id|name|
+---+----+
|  1|   A|
|  2|   B|
|  1|   A|
|  3|   C|
|  2|   B|
|  4|   D|
+---+----+



In [0]:
df1 = df.groupby("id").agg(count("id").alias("duplicate_count")).filter("duplicate_count > 1")
df1.show()

+---+---------------+
| id|duplicate_count|
+---+---------------+
|  1|              2|
|  2|              2|
+---+---------------+



In [0]:
df2 = df1.groupby("id").agg(count("id").alias("total_id_count"))
df2.show()

+---+--------------+
| id|total_id_count|
+---+--------------+
|  1|             1|
|  2|             1|
+---+--------------+



In [0]:
emp_data = [
    (1, "A", 10),
    (2, "B", 20),
    (3, "C", None),
    (4, "D", 30),
    (5, "E", None)
]

dept_data = [
    (10, "HR"),
    (20, "IT")
]

emp_df = spark.createDataFrame(emp_data, ["emp_id", "name", "dept_id"])
dept_df = spark.createDataFrame(dept_data, ["dept_id", "dept_name"])

In [0]:
from pyspark.sql.functions import count,col
df = emp_df.join(dept_df,on="dept_id",how="left").filter(col("dept_id").isNull())
df.show()

+-------+------+----+---------+
|dept_id|emp_id|name|dept_name|
+-------+------+----+---------+
|   NULL|     3|   C|     NULL|
|   NULL|     5|   E|     NULL|
+-------+------+----+---------+



In [0]:
data = [
    (1, 101, 2, 100),   # order_id, product_id, quantity, price
    (2, 101, 1, 100),
    (3, 102, 3, 200),
    (4, 103, 1, 300),
    (5, 102, 2, 200)
]

df = spark.createDataFrame(data, ["order_id", "product_id", "quantity", "price"])
df.show()

+--------+----------+--------+-----+
|order_id|product_id|quantity|price|
+--------+----------+--------+-----+
|       1|       101|       2|  100|
|       2|       101|       1|  100|
|       3|       102|       3|  200|
|       4|       103|       1|  300|
|       5|       102|       2|  200|
+--------+----------+--------+-----+



In [0]:
from pyspark.sql.functions import col, sum
df2 = df.groupby("product_id").agg(sum(col("price") * col("quantity")).alias("total_revenue"))
df2.show()

+----------+-------------+
|product_id|total_revenue|
+----------+-------------+
|       101|          300|
|       102|         1000|
|       103|          300|
+----------+-------------+



In [0]:
data = [
    (1, "A", 3000),
    (2, "B", 5000),
    (3, "C", 4000),
    (4, "D", 5000),
    (5, "E", 6000),
    (6, "F", 2000)
]

df = spark.createDataFrame(data, ["emp_id", "name", "salary"])
df.show()

+------+----+------+
|emp_id|name|salary|
+------+----+------+
|     1|   A|  3000|
|     2|   B|  5000|
|     3|   C|  4000|
|     4|   D|  5000|
|     5|   E|  6000|
|     6|   F|  2000|
+------+----+------+



In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number,col
window_spec = Window.orderBy(col("salary").desc())
df1 = df.withColumn("top_3_salary",row_number().over(window_spec))
df2 = df1.filter("top_3_salary<=3")
df2.show()

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+------+----+------+------------+
|emp_id|name|salary|top_3_salary|
+------+----+------+------------+
|     5|   E|  6000|           1|
|     2|   B|  5000|           2|
|     4|   D|  5000|           3|
+------+----+------+------------+



In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import dense_rank, col

window_spec = Window.orderBy(col("salary").desc())

df.withColumn("rank", dense_rank().over(window_spec)) \
  .filter("rank <= 3") \
  .show()

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+------+----+------+----+
|emp_id|name|salary|rank|
+------+----+------+----+
|     5|   E|  6000|   1|
|     2|   B|  5000|   2|
|     4|   D|  5000|   2|
|     3|   C|  4000|   3|
+------+----+------+----+



In [0]:
purchase_data = [
    (1, "C1"),
    (2, "C2"),
    (3, "C3"),
    (4, "C1"),
    (5, "C4")
]

purchase_df = spark.createDataFrame(purchase_data, ["order_id", "customer_id"])
purchase_df.show()

+--------+-----------+
|order_id|customer_id|
+--------+-----------+
|       1|         C1|
|       2|         C2|
|       3|         C3|
|       4|         C1|
|       5|         C4|
+--------+-----------+



In [0]:
return_data = [
    (101, "C2"),
    (102, "C3")
]

return_df = spark.createDataFrame(return_data, ["return_id", "customer_id"])
return_df.show()

+---------+-----------+
|return_id|customer_id|
+---------+-----------+
|      101|         C2|
|      102|         C3|
+---------+-----------+



In [0]:
from pyspark.sql.functions import dense_rank, col
df = purchase_df.join(return_df,on = "customer_id",how="left").filter(col("return_id").isNull())
df.show()

+-----------+--------+---------+
|customer_id|order_id|return_id|
+-----------+--------+---------+
|         C1|       1|     NULL|
|         C1|       4|     NULL|
|         C4|       5|     NULL|
+-----------+--------+---------+



In [0]:
data = [
    (1, "A", "2022-12-15"),
    (2, "B", "2023-01-10"),
    (3, "C", "2023-06-20"),
    (4, "D", "2024-02-05"),
    (5, "E", "2023-11-30")
]

df = spark.createDataFrame(data, ["id", "name", "hire_date"])
df.show()

+---+----+----------+
| id|name| hire_date|
+---+----+----------+
|  1|   A|2022-12-15|
|  2|   B|2023-01-10|
|  3|   C|2023-06-20|
|  4|   D|2024-02-05|
|  5|   E|2023-11-30|
+---+----+----------+



In [0]:
from pyspark.sql.functions import to_date,year,col
df1 = df.withColumn("hire_date",to_date("hire_date","yyyy-MM-dd"))
df2 = df1.filter(year(col("hire_date")) == 2023).select("id","name").show()


---------------------------------------------------------------------------
NameError                                 Traceback (most recent call last)
File <command-5061226936763543>, line 2
      1 from pyspark.sql.functions import to_date,year,col
----> 2 df1 = df.withColumn("hire_date",to_date("hire_date","yyyy-MM-dd"))
      3 df2 = df1.filter(year(col("hire_date")) == 2023).select("id","name").show()

NameError: name 'df' is not defined

In [0]:
df3 = df.withColumn("hire_date",to_date("hire_date","yyyy-MM-dd"))
df4 = df3.filter(col("hire_date").between("2023-01-01","2024-12-31"))
df4.show()

+---+----+----------+
| id|name| hire_date|
+---+----+----------+
|  2|   B|2023-01-10|
|  3|   C|2023-06-20|
|  4|   D|2024-02-05|
|  5|   E|2023-11-30|
+---+----+----------+



In [0]:

from pyspark.sql.functions import to_date,year,col,max
df3 = df.groupby("id").agg(max("hire_date").alias("latest_order"))
df4 = df1.orderBy(col("latest_order").desc())
df4.limit(3).show()

+---+------------+
| id|latest_order|
+---+------------+
|  4|  2024-02-05|
|  5|  2023-11-30|
|  3|  2023-06-20|
+---+------------+



In [0]:
product_data = [
    (101, "Laptop"),
    (102, "Mobile"),
    (103, "Tablet"),
    (104, "Headphones")
]

product_df = spark.createDataFrame(product_data, ["product_id", "product_name"])
product_df.show()

+----------+------------+
|product_id|product_name|
+----------+------------+
|       101|      Laptop|
|       102|      Mobile|
|       103|      Tablet|
|       104|  Headphones|
+----------+------------+



In [0]:
order_data = [
    (1, 101),
    (2, 102),
    (3, 101),
    (4, 103)
]

order_df = spark.createDataFrame(order_data, ["order_id", "product_id"])
order_df.show()

+--------+----------+
|order_id|product_id|
+--------+----------+
|       1|       101|
|       2|       102|
|       3|       101|
|       4|       103|
+--------+----------+



In [0]:
from pyspark.sql.functions import to_date,year,col,max
lft_join = product_df.join(order_df,on="product_id",how="left").filter(col("order_id").isNull())
lft_join.show()

+----------+------------+--------+
|product_id|product_name|order_id|
+----------+------------+--------+
+----------+------------+--------+



In [0]:
from pyspark.sql.functions import col

lft_join = product_df.join(order_df, on="product_id", how="left") \
    .filter(col("order_id").isNull())

lft_join.show()

+----------+------------+--------+
|product_id|product_name|order_id|
+----------+------------+--------+
|       104|  Headphones|    NULL|
+----------+------------+--------+



In [0]:
data = [
    (1, 101, 2, 100),   # order_id, product_id, quantity, price
    (2, 102, 5, 200),
    (3, 101, 3, 100),
    (4, 103, 1, 300),
    (5, 102, 2, 200),
    (6, 101, 4, 100)
]

df = spark.createDataFrame(data, ["order_id", "product_id", "quantity", "price"])
df.show()

+--------+----------+--------+-----+
|order_id|product_id|quantity|price|
+--------+----------+--------+-----+
|       1|       101|       2|  100|
|       2|       102|       5|  200|
|       3|       101|       3|  100|
|       4|       103|       1|  300|
|       5|       102|       2|  200|
|       6|       101|       4|  100|
+--------+----------+--------+-----+



In [0]:
from pyspark.sql.functions import col
df = df.withColumn("quantity",col("quantity").cast("int"))
df1 = dff.groupBy("product_id").agg(sum("quantity").alias("total_quantity"))
df2 = df1.orderBy(col("total_quantity").desc())
df2.show()

+----------+--------------+
|product_id|total_quantity|
+----------+--------------+
|       101|             9|
|       102|             7|
|       103|             1|
+----------+--------------+



In [0]:
data = [
    (1, "North", 2, 100),
    (2, "South", 1, 200),
    (3, "North", 3, 100),
    (4, "East", 2, 150),
    (5, "South", 4, 200),
    (6, "East", 1, 150)
]

df = spark.createDataFrame(data, ["order_id", "region", "quantity", "price"])
df.show()

+--------+------+--------+-----+
|order_id|region|quantity|price|
+--------+------+--------+-----+
|       1| North|       2|  100|
|       2| South|       1|  200|
|       3| North|       3|  100|
|       4|  East|       2|  150|
|       5| South|       4|  200|
|       6|  East|       1|  150|
+--------+------+--------+-----+



In [0]:
from pyspark.sql.functions import col
df = df.groupby("region").agg(sum(col("quantity") * col("price")).alias("total_revenue"),count("order_id").alias("total_orders"))
df.show()

+------+-------------+------------+
|region|total_revenue|total_orders|
+------+-------------+------------+
| North|          500|           2|
| South|         1000|           2|
|  East|          450|           2|
+------+-------------+------------+



In [0]:
data = [
    (1, "C1"),
    (2, "C1"),
    (3, "C1"),
    (4, "C1"),
    (5, "C1"),
    (6, "C1"),   # C1 → 6 orders

    (7, "C2"),
    (8, "C2"),
    (9, "C2"),

    (10, "C3"),
    (11, "C3"),
    (12, "C3"),
    (13, "C3"),
    (14, "C3"),
    (15, "C3"),  # C3 → 6 orders

    (16, "C4")
]

df = spark.createDataFrame(data, ["order_id", "customer_id"])
df.show()

+--------+-----------+
|order_id|customer_id|
+--------+-----------+
|       1|         C1|
|       2|         C1|
|       3|         C1|
|       4|         C1|
|       5|         C1|
|       6|         C1|
|       7|         C2|
|       8|         C2|
|       9|         C2|
|      10|         C3|
|      11|         C3|
|      12|         C3|
|      13|         C3|
|      14|         C3|
|      15|         C3|
|      16|         C4|
+--------+-----------+



In [0]:
from pyspark.sql.functions import count
df1 = df.groupby("customer_id").agg(count("*").alias("total_count"))
df2 = df1.filter("total_count > 5")
df2.show()

+-----------+-----------+
|customer_id|total_count|
+-----------+-----------+
|         C1|          6|
|         C3|          6|
+-----------+-----------+



In [0]:
from pyspark.sql.functions import count
df = df.groupby("customer_id").agg(count("*").alias("total_count"))
df1 = df.filter("total_count > 5")
df1.show() 

+-----------+-----------+
|customer_id|total_count|
+-----------+-----------+
|         C1|          6|
|         C3|          6|
+-----------+-----------+



In [0]:
data = [
    (1, "A", "2023-07-01","Saturday"),  # Saturday
    (2, "B", "2023-07-03","Monday"),  # Monday
    (3, "C", "2023-07-02","Sunday"),  # Sunday
    (4, "D", "2023-07-05","Wed"),  # Wednesday
    (5, "E", "2023-07-08","Saturday")   # Saturday
]

df = spark.createDataFrame(data, ["emp_id", "name", "hire_date","Day"])
df.show()

+------+----+----------+--------+
|emp_id|name| hire_date|     Day|
+------+----+----------+--------+
|     1|   A|2023-07-01|Saturday|
|     2|   B|2023-07-03|  Monday|
|     3|   C|2023-07-02|  Sunday|
|     4|   D|2023-07-05|     Wed|
|     5|   E|2023-07-08|Saturday|
+------+----+----------+--------+



In [0]:
from pyspark.sql.functions import to_date,dayofweek,dayofmonth
df = df.withColumn("hire_date",to_date("hire_date","yyyy-MM-dd"))
df1 = df.filter(dayofweek("hire_date").isin(1,7))
df1.show()

+------+----+----------+--------+
|emp_id|name| hire_date|     Day|
+------+----+----------+--------+
|     1|   A|2023-07-01|Saturday|
|     3|   C|2023-07-02|  Sunday|
|     5|   E|2023-07-08|Saturday|
+------+----+----------+--------+



In [0]:
df1 = df.select(dayofmonth("hire_date")).show()

---------------------------------------------------------------------------
NameError                                 Traceback (most recent call last)
File <command-4684904139061816>, line 1
----> 1 df1 = df.select(dayofmonth("hire_date")).show()

NameError: name 'dayofmonth' is not defined

In [0]:
data = [
    (1, "2023-01-05", 2, 100),
    (2, "2023-01-15", 1, 200),
    (3, "2023-02-10", 3, 150),
    (4, "2023-02-20", 2, 150),
    (5, "2023-03-05", 4, 100),
    (6, "2023-03-25", 1, 300)
]

df1 = spark.createDataFrame(data, ["order_id", "order_date", "quantity", "price"])
df1.show()

+--------+----------+--------+-----+
|order_id|order_date|quantity|price|
+--------+----------+--------+-----+
|       1|2023-01-05|       2|  100|
|       2|2023-01-15|       1|  200|
|       3|2023-02-10|       3|  150|
|       4|2023-02-20|       2|  150|
|       5|2023-03-05|       4|  100|
|       6|2023-03-25|       1|  300|
+--------+----------+--------+-----+



In [0]:
from pyspark.sql.functions import to_date
df1 = df.withColumn("order_date",to_date("order_date","yyyy-MM-dd"))

---------------------------------------------------------------------------
AttributeError                            Traceback (most recent call last)
File <command-7902322184476376>, line 2
      1 from pyspark.sql.functions import to_date
----> 2 df1 = df.withColumn("order_date",to_date("order_date","yyyy-MM-dd"))

AttributeError: 'GroupedData' object has no attribute 'withColumn'

In [0]:
from pyspark.sql.functions import month, year, sum, count, col,to_date
grouped_df = df1.groupby(year("order_date").alias("year"),month("order_date").alias("month"))
df2 = grouped_df.agg(sum(col("quantity") * col("price").alias("monthly_revenue")),count("order_id").alias("total_count"))
df2.show()

+----+-----+------------------------------------------+-----------+
|year|month|sum((quantity * price AS monthly_revenue))|total_count|
+----+-----+------------------------------------------+-----------+
|2023|    1|                                       400|          2|
|2023|    2|                                       750|          2|
|2023|    3|                                       700|          2|
+----+-----+------------------------------------------+-----------+



In [0]:
print(type(df))

<class 'pyspark.sql.connect.group.GroupedData'>


In [0]:
data = [
    (1, "C1", "2023-01-10"),
    (2, "C1", "2023-02-15"),
    (3, "C1", "2023-03-20"),
    (4, "C1", "2023-04-05"),
    (5, "C1", "2023-05-12"),
    (6, "C1", "2023-06-18"),
    (7, "C1", "2023-07-22"),
    (8, "C1", "2023-08-30"),
    (9, "C1", "2023-09-11"),
    (10, "C1", "2023-10-09"),
    (11, "C1", "2023-11-25"),
    (12, "C1", "2023-12-31"),  # C1 → all 12 months ✅

    (13, "C2", "2023-01-05"),
    (14, "C2", "2023-03-15"),
    (15, "C2", "2023-06-20"),  # C2 → missing months ❌

    (16, "C3", "2023-01-01"),
    (17, "C3", "2023-02-01"),
    (18, "C3", "2023-03-01"),
    (19, "C3", "2023-04-01"),
    (20, "C3", "2023-05-01"),
    (21, "C3", "2023-06-01"),
    (22, "C3", "2023-07-01"),
    (23, "C3", "2023-08-01"),
    (24, "C3", "2023-09-01"),
    (25, "C3", "2023-10-01"),
    (26, "C3", "2023-11-01"),
    (27, "C3", "2023-12-01")   # C3 → all 12 months ✅
]

df = spark.createDataFrame(data, ["order_id", "customer_id", "order_date"])
df.show()

+--------+-----------+----------+
|order_id|customer_id|order_date|
+--------+-----------+----------+
|       1|         C1|2023-01-10|
|       2|         C1|2023-02-15|
|       3|         C1|2023-03-20|
|       4|         C1|2023-04-05|
|       5|         C1|2023-05-12|
|       6|         C1|2023-06-18|
|       7|         C1|2023-07-22|
|       8|         C1|2023-08-30|
|       9|         C1|2023-09-11|
|      10|         C1|2023-10-09|
|      11|         C1|2023-11-25|
|      12|         C1|2023-12-31|
|      13|         C2|2023-01-05|
|      14|         C2|2023-03-15|
|      15|         C2|2023-06-20|
|      16|         C3|2023-01-01|
|      17|         C3|2023-02-01|
|      18|         C3|2023-03-01|
|      19|         C3|2023-04-01|
|      20|         C3|2023-05-01|
+--------+-----------+----------+
only showing top 20 rows


In [0]:
from pyspark.sql.functions import col,year,month,count
df1 = df.filter(year("order_date") == 2023)
df2 = df1.groupBy("customer_id").agg(count(month("order_date")).alias("monthly_count"))
df3 = df2.filter(col("monthly_count") == 12).show()

+-----------+-------------+
|customer_id|monthly_count|
+-----------+-------------+
|         C1|           12|
|         C3|           12|
+-----------+-------------+



In [0]:
data = [
    ("2023-01-01", 100),
    ("2023-01-02", 200),
    ("2023-01-03", 300),
    ("2023-01-04", 400),
    ("2023-01-05", 500)
]

df = spark.createDataFrame(data, ["date", "sales"])

In [0]:
from pyspark.sql.functions import to_date

df = df.withColumn("date", to_date("date", "yyyy-MM-dd"))

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import avg

window_spec = Window.orderBy("date").rowsBetween(-2, 0)

df.withColumn("moving_avg", avg("sales").over(window_spec)).show()

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+----------+-----+----------+
|      date|sales|moving_avg|
+----------+-----+----------+
|2023-01-01|  100|     100.0|
|2023-01-02|  200|     150.0|
|2023-01-03|  300|     200.0|
|2023-01-04|  400|     300.0|
|2023-01-05|  500|     400.0|
+----------+-----+----------+



In [0]:
data = [
    (1, "C1", "2025-10-01"),
    (2, "C1", "2025-11-15"),

    (3, "C2", "2025-01-10"),  # old → churned
    (4, "C2", "2025-02-20"),

    (5, "C3", "2026-03-01"),  # recent → active

    (6, "C4", "2025-08-01"),  # borderline

    (7, "C5", "2024-12-01")   # very old → churned
]

df = spark.createDataFrame(data, ["order_id", "customer_id", "order_date"])
df.show()

+--------+-----------+----------+
|order_id|customer_id|order_date|
+--------+-----------+----------+
|       1|         C1|2025-10-01|
|       2|         C1|2025-11-15|
|       3|         C2|2025-01-10|
|       4|         C2|2025-02-20|
|       5|         C3|2026-03-01|
|       6|         C4|2025-08-01|
|       7|         C5|2024-12-01|
+--------+-----------+----------+



In [0]:
from pyspark.sql.functions import col,year,to_date,add_months,current_date, max
df1 = df.withColumn("order_date",to_date("order_date","yyyy-MM-dd"))
df2 = df1.groupBy("customer_id").agg(max("order_date").alias("last_order_date"))
df3 = df2.filter(col("last_order_date") < add_months(current_date(), -6)).show()                                     

+-----------+---------------+
|customer_id|last_order_date|
+-----------+---------------+
|         C2|     2025-02-20|
|         C4|     2025-08-01|
|         C5|     2024-12-01|
+-----------+---------------+



In [0]:
data = [
    (1, "C1"),
    (2, "C1"),
    (3, "C1"),   # C1 → 3 orders

    (4, "C2"),
    (5, "C2"),   # C2 → 2 orders

    (6, "C3"),
    (7, "C3"),
    (8, "C3"),
    (9, "C3"),   # C3 → 4 orders

    (10, "C4")   # C4 → 1 order
]

df = spark.createDataFrame(data, ["order_id", "customer_id"])
df.show()

+--------+-----------+
|order_id|customer_id|
+--------+-----------+
|       1|         C1|
|       2|         C1|
|       3|         C1|
|       4|         C2|
|       5|         C2|
|       6|         C3|
|       7|         C3|
|       8|         C3|
|       9|         C3|
|      10|         C4|
+--------+-----------+



In [0]:
from pyspark.sql.functions import count,avg,col
df1 = df.groupBy("customer_id").agg(count("order_id").alias("order_count"))
avg_value = df1.agg(avg("order_count").alias("avg_count")).collect()[0][0]
df1.filter(col("order_count") > col("avg_value")).show()


---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-8425345953334178>, line 4
      2 df1 = df.groupBy("customer_id").agg(count("order_id").alias("order_count"))
      3 avg_value = df1.agg(avg("order_count").alias("avg_count")).collect()[0][0]
----> 4 df1.filter(col("order_count") > col("avg_value")).show()

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/dataframe.py:1156, in DataFrame.show(self, n, truncate, vertical)
   1155 def show(self, n: int = 20, truncate: Union[bool, int] = True, vertical: bool = False) -> None:
-> 1156     print(self._show_string(n, truncate, vertical))

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/dataframe.py:909, in DataFrame._show_string(self, n, truncate, vertical)
    892     except ValueError:
    893         raise PySparkTypeError(
    894             errorClass="NOT_BOOL",
    895    

In [0]:
from pyspark.sql.functions import count, avg, col

df1 = df.groupBy("customer_id") \
        .agg(count("order_id").alias("order_count"))

avg_value = df1.agg(avg("order_count").alias("avg_orders")).collect()[0][0]

df1.filter(col("order_count") > avg_value).show()

+-----------+-----------+
|customer_id|order_count|
+-----------+-----------+
|         C1|          3|
|         C3|          4|
+-----------+-----------+



In [0]:
data = [
    (1, "C1", "2023-01-01", 2, 100),
    (2, "C1", "2023-02-01", 1, 100),

    (3, "C2", "2023-01-10", 3, 200),

    (4, "C3", "2023-03-05", 1, 150),
    (5, "C3", "2023-04-01", 2, 150),

    (6, "C4", "2023-05-01", 1, 300)
]

df = spark.createDataFrame(
    data,
    ["order_id", "customer_id", "order_date", "quantity", "price"]
)
df.show()

+--------+-----------+----------+--------+-----+
|order_id|customer_id|order_date|quantity|price|
+--------+-----------+----------+--------+-----+
|       1|         C1|2023-01-01|       2|  100|
|       2|         C1|2023-02-01|       1|  100|
|       3|         C2|2023-01-10|       3|  200|
|       4|         C3|2023-03-05|       1|  150|
|       5|         C3|2023-04-01|       2|  150|
|       6|         C4|2023-05-01|       1|  300|
+--------+-----------+----------+--------+-----+



In [0]:
from pyspark.sql.functions import col,row_number,sum
from pyspark.sql import Window
window_specs = Window.partitionBy("customer_id").orderBy(col("order_id"))
df1 = df.withColumn("rn",row_number().over(window_specs))
df2 = df1.filter(col("rn") == 1)
df3 = df2.withColumn("revenue",col("quantity") * col("price"))
df4 = df3.agg(sum("revenue").alias("total_revenue"))
df4.show()

+-------------+
|total_revenue|
+-------------+
|         1250|
+-------------+



In [0]:
data = [
    ("TV", 10, 30000),
    ("Refrigerator", 5, 25000),
    ("AC", 8, 35000),
    ("Washing Machine", 6, 20000),
    ("Mobile", 20, 15000),
    ("Oven", 7, 10000),
    ("Bluetooth Speaker", 15, 3000)
]

df = spark.createDataFrame(
    data,
    ["product_name", "quantity_sold", "price"]
)

df.show()

In [0]:
from pyspark.sql.functions import col,sum
df1 = df.groupby("product_name").agg(sum(col("price") * col("quantity_sold")).alias("total_revenue")).orderBy(col("total_revenue").desc()).show()


In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import sum
from pyspark.sql.functions import lit
df1 = df.withColumn("revenue",col("price") * col("quantity_sold"))
win = Window.orderBy(col("revenue").desc())
cum_sum = df1.withColumn("cum_revenue",sum("revenue").over(win))
total_rev = cum_sum.agg(sum("revenue"))collect()[0][0]
cum_per = cum_sum.withColumn("cum_percent",col(cum_sum) / lit(total_rev) * 100)
fil = cum_per.filter(col("cum_percent") <= 80).show()



  File <command-5299868743879310>, line 7
    total_rev = cum_sum.agg(sum("revenue"))collect()[0][0]
                                           ^
SyntaxError: invalid syntax


In [0]:
data = [
    ("C1", "2023-01-01"),
    ("C1", "2023-01-05"),
    ("C1", "2023-01-10"),

    ("C2", "2023-02-01"),
    ("C2", "2023-02-10"),

    ("C3", "2023-03-01")  # only one order
]

df = spark.createDataFrame(data, ["customer_id", "order_date"])
df.show()

+-----------+----------+
|customer_id|order_date|
+-----------+----------+
|         C1|2023-01-01|
|         C1|2023-01-05|
|         C1|2023-01-10|
|         C2|2023-02-01|
|         C2|2023-02-10|
|         C3|2023-03-01|
+-----------+----------+



In [0]:
from pyspark.sql.functions import to_date
df = df.withColumn("order_date", to_date("order_date","yyyy-MM-dd"))

In [0]:
from pyspark.sql.functions import *
from pyspark.sql import Window
win = Window.partitionBy("customer_id").orderBy(col("order_date").desc())
df1 = df.withColumn("prev_order_date",lag("order_date").over(win))

In [0]:
df2 = df1.withColumn("date_diff",datediff("order_date","prev_order_date"))

In [0]:
df3 = df2.filter(col("prev_order_date").isNotNull()) \
         .groupBy("customer_id") \
         .agg(avg("date_diff").alias("avg_date_gap")).show()

+-----------+------------+
|customer_id|avg_date_gap|
+-----------+------------+
|         C1|        -4.5|
|         C2|        -9.0|
+-----------+------------+



In [0]:
df3 = df2.groupBy("customer_id").agg(avg("date_diff").alias("avg_date_gap"))
df4 = df3.filter(col("prev_order_date").isnotnull).show()


---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-7482594682114712>, line 2
      1 df3 = df2.groupBy("customer_id").agg(avg("date_diff").alias("avg_date_gap"))
----> 2 df4 = df3.filter(col("prev_order_date").isnotnull).show()

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/dataframe.py:1156, in DataFrame.show(self, n, truncate, vertical)
   1155 def show(self, n: int = 20, truncate: Union[bool, int] = True, vertical: bool = False) -> None:
-> 1156     print(self._show_string(n, truncate, vertical))

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/dataframe.py:909, in DataFrame._show_string(self, n, truncate, vertical)
    892     except ValueError:
    893         raise PySparkTypeError(
    894             errorClass="NOT_BOOL",
    895             messageParameters={
   (...)
    898             },
    899         )
  

In [0]:
data = [
    ("Amit", 85),
    ("Ravi", 92),
    ("Neha", 88),
    ("Priya", 95),
    ("Karan", 90)
]

columns = ["name", "score"]

df = spark.createDataFrame(data, columns)
df.show()

+-----+-----+
| name|score|
+-----+-----+
| Amit|   85|
| Ravi|   92|
| Neha|   88|
|Priya|   95|
|Karan|   90|
+-----+-----+



In [0]:
top3 = df.orderBy(df.score.desc()).limit(3)
top3.show()

+-----+-----+
| name|score|
+-----+-----+
|Priya|   95|
| Ravi|   92|
|Karan|   90|
+-----+-----+



In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

window_spec = Window.orderBy(df.score.desc())

top3 = df.withColumn("rank", row_number().over(window_spec)) \
         .filter("rank <= 3")

top3.show()

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+-----+-----+----+
| name|score|rank|
+-----+-----+----+
|Priya|   95|   1|
| Ravi|   92|   2|
|Karan|   90|   3|
+-----+-----+----+

